In [ ]:
import os
import sys
import gc
import torch
nb_dir = os.getcwd()
if nb_dir not in sys.path:
    sys.path.append(nb_dir)

In [ ]:
%env DB_NAME=
%env DB_USERNAME=
%env DB_PASSWORD=

In [ ]:
from rag_pipeline.main import init_db

db = await init_db()
collection = db.connection["pages"]

cursor = collection.find(no_cursor_timeout=True).batch_size(50)

In [ ]:
import onnxruntime

print(onnxruntime.get_available_providers())

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction, ONNXMiniLM_L6_V2
from sentence_transformers import SentenceTransformer
# sentence_transformer_ef = SentenceTransformerEmbeddingFunction(
#     model_name="Qwen/Qwen3-Embedding-0.6B",
#     device="cuda",
#     normalize_embeddings=True
# )

# class BatchedEmbeddingFunction(SentenceTransformerEmbeddingFunction):
#     def __call__(self, input):
#         embedding = self._model.encode(
#             input,
#             batch_size=16,
#             normalize_embeddings=True
#         ).tolist()
#         torch.cuda.empty_cache()
#         return embedding
# sentence_transformer_ef = BatchedEmbeddingFunction(model_name="Qwen/Qwen3-Embedding-0.6B", device="cuda", model_kwargs={"torch_dtype": "bfloat16"})

ef = ONNXMiniLM_L6_V2(preferred_providers=['CoreMLExecutionProvider'])

chroma_client = chromadb.PersistentClient(path="./db")
# ch_collection = chroma_client.get_or_create_collection(name="articles", embedding_function=sentence_transformer_ef)
ch_collection = chroma_client.get_or_create_collection(name="articles", embedding_function=ef)

In [ ]:
def store_article(title: str, paragraphs: dict):
    for section, content in paragraphs.items():
        text = "".join(content)
        if not text.strip():
            continue
        
        ch_collection.upsert(
            documents=[f"{title}\n{section}\n{text}"],
            metadatas=[{"article": title, "section": section}],
            ids=[f"{title}_{section}"] 
        )

In [ ]:
from rag_pipeline.parser import Parser

i = 1
async for item in cursor:
    print(f"processing chunk N {i} {item["title"]}")
    parser = Parser()
    parsed = parser.get_paragraphs(item["text"])
    if parsed:
        store_article(item["title"], parsed)
    print(f"finished chunk N {i}")
    i += 1

In [ ]:
res = ch_collection.query(query_texts=["Caroline dislikes"], n_results=10)
print(res["documents"][0][7])